## Imports

In [15]:
import duckdb
import pandas as pd
from google.cloud import bigquery
from google.colab import auth
from google.colab import syntax
from google.colab import userdata

pd.options.display.float_format = '{:_.2f}'.format

### Provide your credentials

In [3]:
auth.authenticate_user()
print('Authenticated')

Authenticated


## List projects



In order to query BigQuery, you will need to specify a project ID.  To get a list of project IDs associated with your account, run the following command.

In [4]:
!gcloud projects list --sort-by=projectId

PROJECT_ID                   NAME                    PROJECT_NUMBER  ENVIRONMENT
cool-monolith-286222         Data Science            271608828771
data-science-project-321016  Data Science Project    540766130804
gen-lang-client-0664606651   Default Gemini Project  566992509144


To take a quick anonymous survey, run:
  $ gcloud survey



## Declare the Cloud billing project ID

In [5]:
billing_project_id = userdata.get('bq_billing_project_id')


In [6]:
# Create client object
client = bigquery.Client(project=billing_project_id)


## Get Dataset metadata

### How many data sets are in the bigquery-public-data project?


In [7]:
datasets = client.list_datasets(project="bigquery-public-data")  # Make an API request.
dataset_list = [dataset.dataset_id for dataset in datasets]


In [8]:
len(dataset_list)


382

In [9]:
dataset_list[:10]


['agent_analytic_sample_dataset',
 'america_health_rankings',
 'austin_311',
 'austin_bikeshare',
 'austin_crime',
 'austin_incidents',
 'austin_waste',
 'baseball',
 'bbc_news',
 'biglake-public-nyc-taxi-iceberg.public_data']

### For each data set collect metadata.

In [10]:
datasets = client.list_datasets(project="bigquery-public-data")  # Make an API request.

dataset_metadata = []  # List to hold the data for the DataFrame

for i, dataset in enumerate(datasets):
    # if i >= 3: break

    fdi = f"{dataset.project}.{dataset.dataset_id}"
    dataset_info = client.get_dataset(fdi)
    print(f"{i:03}: {dataset_info.dataset_id}")

    # Append relevant metadata to the list
    dataset_metadata.append({
        "Dataset ID": dataset_info.dataset_id,
        "Description": dataset_info.description,
        "Location": dataset_info.location,
        "Created Time": dataset_info.created,
        "Last Modified Time": dataset_info.modified,
        "Labels": dataset_info.labels,
        "Access Entries": dataset_info.access_entries,
    })

# Create a DataFrame from the list of data
dataset_metadata_df = pd.DataFrame(dataset_metadata)
dataset_metadata_df[:10]

000: agent_analytic_sample_dataset
001: america_health_rankings
002: austin_311
003: austin_bikeshare
004: austin_crime
005: austin_incidents
006: austin_waste
007: baseball
008: bbc_news
009: biglake-public-nyc-taxi-iceberg.public_data
010: bigqueryml_ncaa
011: bitcoin_blockchain
012: blackhole_database
013: blockchain_analytics_ethereum_mainnet_us
014: bls
015: bls_qcew
016: breathe
017: broadstreet_adi
018: catalonian_mobile_coverage
019: catalonian_mobile_coverage_eu
020: census_bureau_acs
021: census_bureau_construction
022: census_bureau_international
023: census_bureau_usa
024: census_opportunity_atlas
025: census_utility
026: cfpb_complaints
027: chicago_crime
028: chicago_taxi_trips
029: clemson_dice
030: cloud_storage_geo_index
031: cms_codes
032: cms_medicare
033: cms_synthetic_patient_data_omop
034: country_codes
035: covid19_aha
036: covid19_covidtracking
037: covid19_ecdc
038: covid19_ecdc_eu
039: covid19_genome_sequence
040: covid19_geotab_mobility_impact
041: covid19_ge

,Dataset ID,Description,Location,Created Time,Last Modified Time,Labels,Access Entries
0,agent_analytic_sample_dataset,Example dataset to demonstrate agent analytics...,US,2026-01-27 05:11:32.156000+00:00,2026-01-29 08:51:44.916000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, specialGroup=proje..."
1,america_health_rankings,America Health Rankings,US,2021-12-30 19:32:54.441000+00:00,2025-07-22 10:45:22.624000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, specialGroup=proje..."
2,austin_311,,US,2017-05-11 18:44:42.673000+00:00,2025-07-22 10:43:48.820000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, specialGroup=proje..."
3,austin_bikeshare,Austin Bikeshare dataset,US,2017-05-16 19:25:37.054000+00:00,2025-07-22 10:44:24.011000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, specialGroup=proje..."
4,austin_crime,Austin Crime dataset,US,2017-05-11 18:29:25.469000+00:00,2025-07-22 10:45:15.608000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, specialGroup=proje..."
5,austin_incidents,,US,2017-05-16 19:25:51.354000+00:00,2025-07-22 10:45:12.508000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, specialGroup=proje..."
6,austin_waste,austin waste and diversion,US,2017-05-16 19:25:20.941000+00:00,2025-07-22 10:45:18.110000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, specialGroup=proje..."
7,baseball,Data Updated On : 2016-10-24\nOverview: This p...,US,2016-10-18 17:08:24.118000+00:00,2025-07-23 09:17:37.113000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, specialGroup=proje..."
8,bbc_news,Data Updated On : 2017-06-07\n,US,2017-06-06 15:21:18.540000+00:00,2026-10-13 07:12:32.569000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, groupByEmail=bigqu..."
9,biglake-public-nyc-taxi-iceberg.public_data,None,us-central1,2025-11-14 17:50:35.493000+00:00,2025-11-14 17:50:35.493000+00:00,{'iceberg_rest_catalog': 'biglake-public-nyc-t...,[]


In [11]:
dataset_metadata_df.drop(columns=["Access Entries"])[:3]


,Dataset ID,Description,Location,Created Time,Last Modified Time,Labels
0,agent_analytic_sample_dataset,Example dataset to demonstrate agent analytics...,US,2026-01-27 05:11:32.156000+00:00,2026-01-29 08:51:44.916000+00:00,"{'dg_data_source': 'public', 'dg_data_context'..."
1,america_health_rankings,America Health Rankings,US,2021-12-30 19:32:54.441000+00:00,2025-07-22 10:45:22.624000+00:00,"{'dg_data_source': 'public', 'dg_data_context'..."
2,austin_311,,US,2017-05-11 18:44:42.673000+00:00,2025-07-22 10:43:48.820000+00:00,"{'dg_data_source': 'public', 'dg_data_context'..."


In [12]:
parquet_file = "bigquery.dataset.metadata.description.parquet"
dataset_metadata_df.drop(columns=["Access Entries"]).to_parquet( parquet_file, index=False)


In [13]:
!ls -l big* --si


-rw-r--r-- 1 root root 64k Nov 13 22:21 bigquery.dataset.metadata.description.parquet


In [16]:
duckdb.query(f"select count(1) from '{parquet_file}' ;")


┌──────────┐
│ count(1) │
│  int64   │
├──────────┤
│      382 │
└──────────┘

In [17]:
duckdb.query(f'''
  select *
  from '{parquet_file}'
  limit 10
  ;
''').df()


,Dataset ID,Description,Location,Created Time,Last Modified Time,Labels
0,agent_analytic_sample_dataset,Example dataset to demonstrate agent analytics...,US,2026-01-27 05:11:32.156000+00:00,2026-01-29 08:51:44.916000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
1,america_health_rankings,America Health Rankings,US,2021-12-30 19:32:54.441000+00:00,2025-07-22 10:45:22.624000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
2,austin_311,,US,2017-05-11 18:44:42.673000+00:00,2025-07-22 10:43:48.820000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
3,austin_bikeshare,Austin Bikeshare dataset,US,2017-05-16 19:25:37.054000+00:00,2025-07-22 10:44:24.011000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
4,austin_crime,Austin Crime dataset,US,2017-05-11 18:29:25.469000+00:00,2025-07-22 10:45:15.608000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
5,austin_incidents,,US,2017-05-16 19:25:51.354000+00:00,2025-07-22 10:45:12.508000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
6,austin_waste,austin waste and diversion,US,2017-05-16 19:25:20.941000+00:00,2025-07-22 10:45:18.110000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
7,baseball,Data Updated On : 2016-10-24\nOverview: This p...,US,2016-10-18 17:08:24.118000+00:00,2025-07-23 09:17:37.113000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
8,bbc_news,Data Updated On : 2017-06-07\n,US,2017-06-06 15:21:18.540000+00:00,2026-10-13 07:12:32.569000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
9,biglake-public-nyc-taxi-iceberg.public_data,None,us-central1,2025-11-14 17:50:35.493000+00:00,2025-11-14 17:50:35.493000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."


In [ ]:
stop


## Push parquet to AWS S3


In [ ]:
import os


In [ ]:
%%capture install_output
%%bash
pip install -U awscli


In [ ]:
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('aws_key')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('aws_secret')
os.environ['AWS_DEFAULT_REGION'] = 'us-west-1'


In [ ]:
f"s3://ddc-datascience/{parquet_file}"

In [ ]:
!aws s3 cp {parquet_file} s3://ddc-datascience/{parquet_file}


In [ ]:
url = f"https://ddc-datascience.s3.amazonaws.com/{parquet_file}"
url


In [ ]:
duckdb.query(f"select count(1) as 'number of data sets' from '{url}' ;").df()


In [ ]:
duckdb.query(f"select * from '{url}' limit 1;").df()


In [ ]:
df["Last Modified Time"].max()


In [ ]:
df = duckdb.query(f"select * from '{url}';").df()
df.shape


In [ ]:
import textwrap

for i, row in df.iterrows():
  print( f"=== {i:03}" )
  print( row["Dataset ID"] )
  description = row.get( "Description", "" )
  for line in description.splitlines():
    print( textwrap.fill( line, width = 150 ) )
  print()


In [ ]:
## trying to understand
datasets = client.list_datasets(project="bigquery-public-data")  # Make an API request.
for i, dataset in enumerate(datasets):
    if i >= 1: break
[ x for x in dir(dataset) if not x.startswith("_") ]

full_dataset = client.get_dataset(dataset.reference)
df = pd.DataFrame([ ( x, type(getattr(full_dataset, x)).__name__ ) for x in dir(full_dataset) if not x.startswith("_") ]).sort_values(by=[1,0])

import yaml
print(yaml.dump(full_dataset.to_api_repr(), sort_keys=False, default_flow_style=False))


In [ ]:
## trying to understand
datasets = client.list_datasets(project="bigquery-public-data")  # Make an API request.

# dataset_metadata = []  # List to hold the data for the DataFrame

for i, dataset in enumerate(datasets):
    if i >= 3: break

    fdi = f"{dataset.project}.{dataset.dataset_id}"
    dataset_info = client.get_dataset(fdi)

    print(f"{i:03}: {dataset_info.dataset_id}")
    print(f"{i:03}: {dataset.dataset_id}")
    foo = dir(dataset)
    print(len(foo))

#     # Append relevant metadata to the list
#     dataset_metadata.append({
#         "Dataset ID": dataset_info.dataset_id,
#         "Description": dataset_info.description,
#         "Location": dataset_info.location,
#         "Created Time": dataset_info.created,
#         "Last Modified Time": dataset_info.modified,
#         "Labels": dataset_info.labels,
#         "Access Entries": dataset_info.access_entries,
#     })

# # Create a DataFrame from the list of data
# dataset_metadata_df = pd.DataFrame(dataset_metadata)
# dataset_metadata_df[:10]



### For each data set, get table information.

In [ ]:
dataset_id
tables = client.list_tables(dataset)
try:
  for table in tables:
    print("hi")
except:
  print("nope")


In [ ]:
datasets = client.list_datasets(project="bigquery-public-data")  # Make an API request.

data = []  # List to hold the data for the DataFrame

for i, dataset in enumerate(datasets):
  # if i <= 68: continue
  # if i >= 75: break

  dataset_id = dataset.dataset_id
  print(f"{i:03}: {dataset_id}")
  tables = client.list_tables(dataset)  # Make an API request.
  try:
    for table in tables:
      table_ref = client.get_table(table)  # Make an API request.
      table_id = table_ref.table_id
      num_rows = table_ref.num_rows
      num_columns = len(table_ref.schema)
      size_mb = round(table_ref.num_bytes / (1000 ** 2),2)
      modified = table_ref.modified

      # Append the data to the list
      data.append({
        "Dataset ID": dataset_id,
        'Table': table_id,
        'Number of Rows': num_rows,
        'Number of Columns': num_columns,
        'Size in MB': size_mb,
        'Last Modified': modified,
    })
  except:
    data.append({
      "Dataset ID": dataset_id,
      'Table': None,
      'Number of Rows': 0,
      'Number of Columns': 0,
      'Size in MB': None,
      'Last Modified': None,
    })


# Create a DataFrame from the list of data
table_df = pd.DataFrame(data)
table_df[:10]

In [ ]:


# stop



## Create parquet from data frame


In [ ]:
parquet_file = "bigquery.dataset.metadata.parquet"
table_df.to_parquet( parquet_file, index=False)


In [ ]:
!ls -l big* --si


In [ ]:
duckdb.query(f"select count(1) from '{parquet_file}' ;")


In [ ]:
import duckdb
duckdb.query(f"select * from '{parquet_file}' limit 10;").df()


## Push parquet to AWS S3


In [ ]:
import os


In [ ]:
%%capture install_output
%%bash
pip install awscli


In [ ]:
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('aws_key')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('aws_secret')
os.environ['AWS_DEFAULT_REGION'] = 'us-west-1'


In [ ]:
!aws sts get-caller-identity

In [ ]:
!aws s3 cp {parquet_file} s3://ddc-datascience/{parquet_file}


In [ ]:
old_parquet_file = "bigquery.dataset.metadata.parquet"
parquet_file = "bigquery.dataset.metadata.tables.parquet"

!aws s3 mv s3://ddc-datascience/{old_parquet_file} s3://ddc-datascience/{parquet_file}


In [ ]:
url = f"https://ddc-datascience.s3.amazonaws.com/{parquet_file}"
duckdb.query(f"select 1 from '{url}' limit 1 ;")



In [ ]:
parquet_file = "bigquery.dataset.metadata.tables.parquet"
url = f"https://ddc-datascience.s3.amazonaws.com/{parquet_file}"
duckdb.query(f"select * from '{url}' limit 1 ;")


In [ ]:
import duckdb

parquet_file = "bigquery.dataset.metadata.tables.parquet"
url = f"https://ddc-datascience.s3.amazonaws.com/{parquet_file}"
duckdb.query(f"select * from '{url}' limit 1 ;")

In [ ]:
import duckdb

url_md_desc = f"https://ddc-datascience.s3.amazonaws.com/bigquery.dataset.metadata.description.parquet"
duckdb.query(f"select * from '{url_md_desc}' limit 1 ;").df()


In [ ]:

url_md_tables = f"https://ddc-datascience.s3.amazonaws.com/bigquery.dataset.metadata.tables.parquet"
duckdb.query(f"select * from '{url_md_tables}' limit 1 ;").df()


### Copy pickle to Google Drive

In [ ]:
from google.colab import drive
drive.mount('drive')


In [ ]:
!ls -la


In [ ]:
!ls -la drive/MyDrive/tmp/


In [ ]:
!cp bigquery.dataset.metadata.pkl drive/MyDrive/tmp/


## Load pickle archive from AWS

In [ ]:
url = "https://ddc-datascience.s3.amazonaws.com/bigquery.dataset.metadata.pkl"
df = pd.read_pickle( url )
df.shape

In [ ]:
df

### Number of tables in a data set

In [ ]:
df["Dataset"].value_counts().rename('Number of Tables')

### Size of data sets ( in GB ).

In [ ]:
(
  df
  .groupby(["Dataset"])["Size in MB"]
  .sum()/1000
).sort_values(ascending=False).rename('Size in GB')


### Size of tables ( in GB ) in each data set.

In [ ]:
( (df.groupby(["Dataset", "Table"])["Size in MB"].sum()/1000)
  .sort_values(ascending=False)
  .rename('Size in GB')
  .reset_index()
)